# Exploration des données sources — modélisation OMEGA BI (C13)

**Compétence couverte : C13 — Modéliser un entrepôt de données**
**Épreuve associée : E5**

Ce notebook reproduit, avec sortie réelle capturée, les analyses
exploratoires qui ont directement motivé des choix de
[`modelisation_omega_bi.md`](../docs/architecture/modelisation_omega_bi.md) :

1. `stocks.csv` est un instantané unique, pas une série temporelle — a
   motivé le choix de modéliser `Fait_Stock` comme une *periodic
   snapshot fact table* prête pour un historique futur (§5.2).
2. `produits.categorie` est parfaitement corrélée à `client_id` (10
   produits par client, une seule catégorie chacun) — contexte pour le
   choix de flocon `Dim_Produit` → `Dim_Categorie` (§6.3/§7).
3. Cardinalités et cohérence de `historique_expeditions` (clients,
   entrepôts, catégories, statuts) — a confirmé que le rapprochement
   textuel `client`/`entrepot` est fiable sur ce jeu de données, malgré
   l'absence de contrainte (§5.1, §8).
4. Rappel des 3 données personnelles du registre RGPD de la base de
   travail (`registre_rgpd.md`) — justifie l'exclusion RGPD *by design*
   de `transporteurs.contact` de `Dim_Transporteur` (§6.5).
5. Correspondance exacte `expeditions.transporteur` ↔ `transporteurs.nom`
   — a motivé la clarification apportée à §5.1 sur le rapprochement
   textuel côté FluxPro (remarque de relecture du 28/08/2026).

Chaque section reprend une commande effectivement lancée pendant la
conception du modèle, pas une réécriture a posteriori.


## 1. `stocks.csv` : instantané unique, pas une série temporelle

Vérifie que les 90 lignes de `stocks.csv` couvrent chaque couple
(entrepôt, produit) une seule fois — donc qu'aucune profondeur
temporelle n'existe encore dans le jeu de données pédagogique, même si
la colonne `date_maj` varie.


In [1]:
import csv
from pathlib import Path

RAW = Path("..") / "data" / "raw"

with open(RAW / "stocks.csv") as f:
    rows = list(csv.DictReader(f))

print("rows:", len(rows))
print("distinct entrepot_id:", len(set(r["entrepot_id"] for r in rows)))
print("distinct produit_id:", len(set(r["produit_id"] for r in rows)))
print("distinct date_maj:", sorted(set(r["date_maj"] for r in rows)))
pairs = set((r["entrepot_id"], r["produit_id"]) for r in rows)
print("distinct (entrepot,produit) pairs:", len(pairs))


rows: 90
distinct entrepot_id: 3
distinct produit_id: 30
distinct date_maj: ['2026-07-27', '2026-07-28', '2026-07-29', '2026-07-30', '2026-07-31', '2026-08-01']
distinct (entrepot,produit) pairs: 90


**Lecture** : 90 lignes = 3 entrepôts × 30 produits, une seule ligne
par couple — confirmé : un instantané unique, malgré 6 dates de mise à
jour différentes. `Fait_Stock` est donc conçu en §5.2 comme une
*periodic snapshot fact table* au grain (entrepôt, produit, date), prête
à recevoir un nouvel instantané à chaque exécution future de l'ETL
(C15), sans être limitée par cette absence actuelle d'historique.


## 2. `produits.categorie` : corrélation parfaite avec le client

Vérifie si la catégorie d'un produit est réductible à son client
propriétaire — pertinent pour juger si `Dim_Categorie` apporte une
information vraiment indépendante de `Dim_Client`, ou si elle n'est
qu'un doublon.


In [2]:
with open(RAW / "produits.csv") as f:
    produits = list(csv.DictReader(f))

from collections import Counter

print("categories:", Counter(r["categorie"] for r in produits))
print("distinct clients (produits.client_id):", sorted(set(r["client_id"] for r in produits)))

par_client = {}
for r in produits:
    par_client.setdefault(r["client_id"], set()).add(r["categorie"])
print("categories distinctes par client_id:", par_client)


categories: Counter({'Pieces auto': 10, 'Alimentaire': 10, 'Textile': 10})
distinct clients (produits.client_id): ['1', '2', '3']
categories distinctes par client_id: {'1': {'Pieces auto'}, '2': {'Alimentaire'}, '3': {'Textile'}}


**Lecture** : chaque client (1, 2, 3) n'a qu'une seule catégorie de
produits (10 produits chacun) — la catégorie est donc, sur ce jeu de
données, réductible au client. Ce n'est **pas** une raison suffisante
pour fusionner `Dim_Categorie` dans `Dim_Client` : la justification
retenue en §6.3 de `modelisation_omega_bi.md` tient à
`historique_expeditions`, qui porte une catégorie sans porter de SKU —
`Dim_Categorie` reste nécessaire comme dimension à grain réduit pour ce
cas, indépendamment de cette corrélation observée côté FluxPro.


## 3. `historique_expeditions` : cardinalités et cohérence

Vérifie que les valeurs texte libre `client`/`entrepot`/`categorie_produit`
de l'historique correspondent bien aux référentiels connus (3 clients,
3 entrepôts), et donne la distribution des statuts.


In [3]:
with open(RAW / "historique" / "omega_historique_expeditions.csv") as f:
    historique = list(csv.DictReader(f))

print("lignes:", len(historique))
print("clients:", Counter(r["client"] for r in historique))
print("entrepots:", Counter(r["entrepot"] for r in historique))
print("categorie_produit:", Counter(r["categorie_produit"] for r in historique))
print("statut:", Counter(r["statut"] for r in historique))


lignes: 25000
clients: Counter({'MedioTex': 8412, 'FreshMarket': 8297, 'NordDrive': 8291})
entrepots: Counter({'Lyon': 8376, 'Marseille': 8361, 'Lille': 8263})
categorie_produit: Counter({'Textile': 8412, 'Alimentaire': 8297, 'Pieces auto': 8291})
statut: Counter({'Livree': 21251, 'Retardee': 2732, 'Incident': 1017})


In [4]:
with open(RAW / "clients.csv") as f:
    noms_clients_connus = set(r["nom"] for r in csv.DictReader(f))
with open(RAW / "entrepots.csv") as f:
    villes_entrepots_connues = set(r["ville"] for r in csv.DictReader(f))

clients_historique = set(r["client"] for r in historique)
entrepots_historique = set(r["entrepot"] for r in historique)

clients_orphelins = clients_historique - noms_clients_connus
entrepots_orphelins = entrepots_historique - villes_entrepots_connues
print("clients historique hors referentiel connu:", clients_orphelins or "aucun")
print("entrepots historique hors referentiel connu:", entrepots_orphelins or "aucun")


clients historique hors referentiel connu: aucun
entrepots historique hors referentiel connu: aucun


**Lecture** : les 3 clients et les 3 entrepôts de l'historique
correspondent exactement aux référentiels FluxPro connus, sans valeur
orpheline. Confirme, pour ce jeu de données précis, la fiabilité du
rapprochement textuel décrit en §5.1/§8 de `modelisation_omega_bi.md` —
tout en rappelant que ce n'est pas garanti par une contrainte, donc à
contrôler explicitement lors du chargement (C15), pas supposé fiable
par défaut en production.


## 4. Rappel des données personnelles du registre RGPD

Relit `registre_rgpd.md` (base de travail, Bloc 2) pour retrouver les
colonnes identifiées comme données personnelles — sert de référence
directe à l'exclusion RGPD *by design* de `transporteurs.contact` de
`Dim_Transporteur` (§6.5 de `modelisation_omega_bi.md`).


In [5]:
import re

registre = Path("..") / "docs" / "architecture" / "registre_rgpd.md"
lignes_donnees_perso = [
    ligne for ligne in registre.read_text().splitlines()
    if re.search(r"chauffeur|adresse_livraison|contact", ligne) and ligne.strip().startswith("|")
]
for ligne in lignes_donnees_perso[:3]:
    print(ligne)


| `chauffeur` | `tournees` | Nom du conducteur (ex. « Yanis L. ») | Salarié d'un transporteur partenaire |
| `adresse_livraison` | `livraisons` | Adresse du destinataire | Client final (destinataire de la marchandise) |
| `contact` | `transporteurs` | Contact professionnel (ex. « contact@rapidfret.example ») | Contact d'entreprise, faible sensibilité |


**Lecture** : trois colonnes personnelles identifiées dans la base de
travail — `tournees.chauffeur`, `livraisons.adresse_livraison`,
`transporteurs.contact`. Aucune des trois n'est mobilisée par une
dimension ou une mesure de l'entrepôt OMEGA BI : `Dim_Transporteur` ne
retient que `nom` (§6.5), et `chauffeur`/`adresse_livraison` ne sont
utilisées par aucun fait ni dimension de ce modèle.


## 5. `expeditions.transporteur` ↔ `transporteurs.nom` : correspondance textuelle

Vérification demandée en relecture (28/08/2026) : `expeditions.transporteur`
(FluxPro) est un texte libre, pas une clé étrangère vers `transporteurs.id`
— à quel point peut-on compter sur une jointure textuelle exacte vers
`Dim_Transporteur.nom` ?


In [6]:
import json as jsonlib

with open(RAW / "expeditions.csv") as f:
    transporteurs_expeditions = set(r["transporteur"] for r in csv.DictReader(f))

with open(Path("..") / "api-mock" / "fixtures" / "transporteurs.json") as f:
    transporteurs_dim = set(t["nom"] for t in jsonlib.load(f))

print("Valeurs cote expeditions.transporteur (FluxPro):", transporteurs_expeditions)
print("Valeurs cote transporteurs.nom (TransFlow, source de Dim_Transporteur):", transporteurs_dim)
print("Correspondance exacte ?", transporteurs_expeditions == transporteurs_dim)
print("Ecart:", transporteurs_expeditions.symmetric_difference(transporteurs_dim) or "aucun")


Valeurs cote expeditions.transporteur (FluxPro): {'EcoRoute', 'TransUnion Logistique', 'RapidFret'}
Valeurs cote transporteurs.nom (TransFlow, source de Dim_Transporteur): {'EcoRoute', 'TransUnion Logistique', 'RapidFret'}
Correspondance exacte ? True
Ecart: aucun


**Lecture** : correspondance exacte confirmée, aucune variante
d'orthographe sur ce jeu de données. Comme pour le rapprochement
client/entrepôt de l'historique (§3), ceci n'est pas garanti par une
contrainte — `modelisation_omega_bi.md` §5.1 documente désormais
explicitement ce mécanisme de jointure textuelle et le contrôle qualité
à prévoir en C15.


## 6. Rappel : `commandes.statut` (contexte `Fait_Commande`)

Distribution des statuts de commande FluxPro, pour contexte sur le choix
(§5.3) de dénormaliser `statut_commande` comme attribut dégénéré plutôt
que comme dimension séparée.


In [7]:
with open(RAW / "commandes.csv") as f:
    commandes = list(csv.DictReader(f))

print("commandes:", len(commandes))
print("statut:", Counter(r["statut"] for r in commandes))


commandes: 1400
statut: Counter({'Livree': 761, 'Expediee': 339, 'En preparation': 225, 'Annulee': 75})


**Lecture** : 4 valeurs de statut seulement, sans attribut
additionnel — confirme qu'une dimension dédiée n'apporterait rien de
plus qu'un attribut dégénéré sur `Fait_Commande` (§5.3).
